In [1]:
import sys
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from codecarbon import EmissionsTracker

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(".env")

from src.data.loader import Loader
from src.features.vectorstore import VectorStore
from src.models.classifier import Classifier
from src.evaluation.logging import log_predictions_as_artifact, log_configuration
from src.evaluation.metrics import evaluate

import mlflow
import yaml

/mnt/data_ml/workspace/flex-care/venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# config
with open("../references/configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

mlflow.set_tracking_uri("http://localhost:8080")

mlflow.set_experiment(cfg["project_name"])
mlflow.start_run(run_name=f'{cfg["model"]["name"]}-{cfg["dataset"]["name"]}-{cfg["model"]["n_output"]}')

log_configuration(cfg)

In [3]:
# data
loader  = Loader()
dataset = loader.load_dataset(cfg["dataset"]["name"])

In [4]:
# vectorstore
index_path = Path(cfg["retrieval"]["index_path"])
vs = VectorStore(embeddings=cfg["retrieval"]["embeddings"], docs=loader.docs)

if index_path.exists():
    vs.load(index_path)
else:
    vs.build(chunk_size=cfg["retrieval"]["chunk_size"], save_path=index_path)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [5]:
# inference

tracker = EmissionsTracker(project_name=cfg["project_name"])
tracker.start()

classifier = Classifier(**cfg["model"])
results = classifier.predict(
    vs      = vs,
    dataset = dataset,
    k       = cfg["retrieval"]["k"],
    every_represented_candidate = cfg["retrieval"]["every_represented_candidate"],
)

emissions = tracker.stop()
mlflow.log_metric("co2_kg", emissions)

log_predictions_as_artifact(results, dataset)

[codecarbon WARNING @ 08:47:51] Multiple instances of codecarbon are allowed to run at the same time.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Predicting: 100%|██████████| 132/132 [04:10<00:00,  1.89s/it]


In [6]:
# metrics
scalar_metrics = evaluate(results)
mlflow.log_metrics(scalar_metrics)

In [7]:
mlflow.end_run()

🏃 View run prithivMLmods/Qwen-UMLS-7B-Instruct-motu_enriched.csv-3 at: http://localhost:8080/#/experiments/429929398258185050/runs/77d183cdd33a4fdca797e6549eac6ded
🧪 View experiment at: http://localhost:8080/#/experiments/429929398258185050
